# 23 (versión 2) · Benchmark externo reanudable y seguro

**Corrección del bloqueo:** el registro anterior contiene 53 bloques, uno de 2,061.9 s. La versión 2 limita cada pasada a **1024 tokens** y, en documentos más extensos, conserva todas las partes mediante fragmentación con solapamiento y promedio ponderado de embeddings normalizados. **No se recorta ni omite texto**, pero el promedio de fragmentos **es otra estrategia de representación** y las métricas V1 y V2 no son directamente intercambiables.

- Mantiene exactamente los mismos cinco modelos, revisiones, datos públicos, preguntas, etiquetas y ponderaciones del benchmark original.
- Conserva `benchmark_finalistas_v1.json` sin modificar; registra la condición nueva en `benchmark_finalistas_v2.json`.
- Reutiliza los vectores V1 **únicamente para documentos que caben íntegros en 1024 tokens**, verificando el hash de cada bloque y la condición original. Documentos extensos se vuelven a representar íntegramente.
- Reutiliza descargas y archivos de Hugging Face; no hace falta adjuntar nada más.
- Guarda cada 32 documentos y muestra el avance cada 256; puede reanudarse después de una interrupción.

**Entorno:** kernel `entorno_tesis`, Windows con RTX 4070 SUPER. **No ejecutar simultáneamente otros modelos GPU.** Resultados de esta versión son independientes y deben compararse únicamente con otros modelos evaluados en la misma condición V2.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter, sleep
from collections import defaultdict
from itertools import combinations
import gc, hashlib, importlib.metadata, io, json, platform, re, sys, zipfile
import numpy as np
import pandas as pd
import requests
from IPython.display import display

NOMBRE_LIBRETA = "23_reanudacion_segura_benchmark.ipynb"
VERSION = "BENCHMARK_EXTERNO_FINALISTAS_V2_SEGMENTOS"
LIMITE_FRAGMENTO = 1024
SOLAPAMIENTO_TOKENS = 96
TAM_BLOQUE = 32
SEMILLA = 2026
KS = [5, 10, 20, 100]
BOOTSTRAPS = 10000
ALFA_FAMILIA = 0.05
MARGEN_PRACTICO = 0.01
PESOS = {"scifact": .2, "scidocs": .2, "nfcorpus": .2,
         "miracl_es_hn": .2, "xquad_es_en": .2}
MODELOS = {
    "qwen3_4b": dict(repo="Qwen/Qwen3-Embedding-4B",
        revision="5cf2132abc99cad020ac570b19d031efec650f2b", dtype="bfloat16", dim=2560,
        query="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:",
        document="", prompt_name="query", max_tokens=32768, batch=1, remote=False),
    "nemotron3_1b": dict(repo="nvidia/Nemotron-3-Embed-1B-BF16",
        revision="c0c9fea93ea424587517f2c59e20db9f1d6bf615", dtype="bfloat16", dim=2048,
        query="query: ", document="passage: ", prompt_name="query", max_tokens=32768, batch=4, remote=False),
    "arctic_l_v2": dict(repo="Snowflake/snowflake-arctic-embed-l-v2.0",
        revision="ac6544c8a46e00af67e330e85a9028c66b8cfd9a", dtype="float32", dim=1024,
        query="query: ", document="", prompt_name="query", max_tokens=8192, batch=4, remote=False),
    "pplx_4b_bf16": dict(repo="perplexity-ai/pplx-embed-v1-4b",
        revision="06456497a00540a582918fe8dcd3a5eabb207772", dtype="bfloat16", dim=2560,
        query="", document="", prompt_name=None, max_tokens=32768, batch=1, remote=True),
    "harrier_0_6b": dict(repo="microsoft/harrier-oss-v1-0.6b",
        revision="f9b9dc8d367d443f2479d27aa5d8d2850c0774ee", dtype="bfloat16", dim=1024,
        query="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ",
        document="", prompt_name="web_search_query", max_tokens=32768, batch=4, remote=False),
}

carpetas = [p / s for p in (Path.cwd(), *Path.cwd().parents) for s in
            (".", "07_RAG/Pruebas_RAG/Benchmark_Externo", "Pruebas_RAG/Benchmark_Externo",
             "Benchmark_Externo", "07_RAG/Pruebas_RAG", "Pruebas_RAG")]
BASE = next((p for p in carpetas if (p / NOMBRE_LIBRETA).is_file()), None)
if BASE is None:
    raise FileNotFoundError(f"Guarda la libreta como {NOMBRE_LIBRETA} y abre su carpeta.")
BASE = BASE.resolve()
CACHE = BASE / "_cache_externo_v1"
CACHE.mkdir(exist_ok=True)
SALIDA = BASE / "benchmark_finalistas_v2.json"
SALIDA_ANTERIOR = BASE / "benchmark_finalistas_v1.json"

def exigir(ok, mensaje):
    if not ok: raise ValueError(mensaje)

def ahora(): return datetime.now(timezone.utc).isoformat()

def firma(x):
    return hashlib.sha256(json.dumps(x, ensure_ascii=False, sort_keys=True,
                          separators=(",", ":"), allow_nan=False).encode()).hexdigest()

def hash_archivo(p, algoritmo="sha256"):
    h = hashlib.new(algoritmo)
    with Path(p).open("rb") as f:
        for b in iter(lambda: f.read(8 * 1024 * 1024), b""): h.update(b)
    return h.hexdigest()

def guardar(p, x):
    tmp = Path(str(p) + ".tmp")
    tmp.write_text(json.dumps(x, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    tmp.replace(p)

notebook = json.loads((BASE / NOMBRE_LIBRETA).read_text(encoding="utf-8"))
CODIGO_SHA = firma(["".join(c["source"]) for c in notebook["cells"] if c["cell_type"] == "code"])
PROTOCOLO = dict(version=VERSION, semilla=SEMILLA, modelos=MODELOS, pesos=PESOS, cortes=KS,
    n_bootstraps=BOOTSTRAPS, alfa_familia=ALFA_FAMILIA, margen_practico=MARGEN_PRACTICO,
    codigo_sha256=CODIGO_SHA, ajuste_de_pesos=False, entrenamiento=False,
    busqueda="exhaustiva float32 CPU; desempate por ID de documento lexicográfico",
    documentos="BEIR/MIRACL: título + salto de línea + texto; XQuAD: contexto sin respuesta añadida",
    max_tokens="cada pasada <= 1024 tokens con verificación; el texto largo se fragmenta sin omitir contenido",
    procesamiento_largo=dict(max_tokens_por_pasada=LIMITE_FRAGMENTO, solape_tokens=SOLAPAMIENTO_TOKENS,
        bloque_documentos=TAM_BLOQUE, agregacion="media ponderada por nuevos tokens de los vectores L2 de fragmentos; normalización L2 final",
        sesgo="representación V2; no equivale al embedding de texto íntegro V1"),
    ndcg="ganancia lineal igual al grado publicado; nDCG@10 principal",
    desconocidos="convención de benchmark: aporte 0 al score; no juicio nuevo de irrelevancia",
    muestreo="todos los queries y documentos de los splits declarados; MIRACL usa corpus público reducido HN",
    estadistica="bootstrap pareado por bloques, 10000; CI simultáneos Bonferroni para 10 pares",
    independencia="de la evaluación interna; exposición en preentrenamiento no descartada",
    decision="no seleccionar antes de completar todos; no equiparar ausencia de significancia a equivalencia")
FIRMA_PROTOCOLO = firma(PROTOCOLO)
lock = CACHE / "protocolo_v2.json"
if lock.exists():
    exigir(json.loads(lock.read_text(encoding="utf-8"))["firma"] == FIRMA_PROTOCOLO,
           "Cambió el protocolo/código. No mezcles condiciones; usa una carpeta nueva para otra versión.")
else: guardar(lock, dict(firma=FIRMA_PROTOCOLO, fijado_utc=ahora(), protocolo=PROTOCOLO))
if SALIDA.exists():
    R = json.loads(SALIDA.read_text(encoding="utf-8"))
    exigir(R["firma_protocolo"] == FIRMA_PROTOCOLO, "La salida pertenece a otro protocolo.")
    for modelo in R.get("modelos", {}).values():
        for run in modelo.get("tareas", {}).values():
            exigir(firma(run["datos"]) == run["firma"], "Resultado de tarea alterado.")
else:
    R = dict(version=VERSION, firma_protocolo=FIRMA_PROTOCOLO, protocolo=PROTOCOLO,
             creado_utc=ahora(), estado="preparado", datasets={}, modelos={}, incidencias=[])
    guardar(SALIDA, R)
ANTERIOR = json.loads(SALIDA_ANTERIOR.read_text(encoding="utf-8")) if SALIDA_ANTERIOR.exists() else None
if ANTERIOR:
    exigir(ANTERIOR.get("version") == "BENCHMARK_EXTERNO_FINALISTAS_V1", "Resultado V1 anterior inesperado")
    exigir(ANTERIOR["protocolo"]["modelos"] == MODELOS, "No coincide la configuración V1 de los modelos")
print("Carpeta:", BASE)
print("Protocolo V2 separado de V1; descargas y checkpoints V1 son solo de lectura.")

## 1. Descargar y validar los datos externos

Se usan los test publicados de **SciFact, SCIDOCS y NFCorpus**, el **dev español de MIRACL Hard Negatives** con control de correspondencia entre corpus y juicios (se excluye toda consulta a la que le falte algún documento positivo, sin modificar etiquetas) y **XQuAD español → inglés**. XQuAD inglés → inglés y español → español quedan como diagnósticos, sin peso adicional. MIRACL HN es un corpus reducido por los autores de MTEB, no toda Wikipedia.

En XQuAD la etiqueta es el pasaje de origen con respuesta anotada; no se declara que sea el único pasaje semánticamente válido. Se comprueban alineación ES/EN e intervalos de respuesta, pero las respuestas nunca se agregan al texto de búsqueda.

Las descargas tienen verificación de integridad. MIRACL fija su revisión pública en el primer acceso y después la conserva. Si falta una fuente o un positivo, se detiene; no sustituye datos ni fabrica etiquetas.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

FUENTES = {}
TAREAS = {}
SESION_HTTP = requests.Session()
SESION_HTTP.headers["User-Agent"] = "TESIS-ISBD-external-benchmark/1.0"

def descargar(url, nombre, md5=None, sha256=None):
    ruta = CACHE / "descargas" / nombre
    ruta.parent.mkdir(exist_ok=True)
    if not ruta.exists():
        tmp = Path(str(ruta) + ".part")
        for intento in range(3):
            try:
                with SESION_HTTP.get(url, stream=True, timeout=(30, 120)) as resp:
                    resp.raise_for_status()
                    with tmp.open("wb") as f:
                        for b in resp.iter_content(1024 * 1024):
                            if b: f.write(b)
                tmp.replace(ruta)
                break
            except (requests.RequestException, OSError):
                if intento == 2: raise
                sleep(2 ** intento)
    if md5: exigir(hash_archivo(ruta, "md5") == md5, f"MD5 distinto: {nombre}")
    h = hash_archivo(ruta)
    if sha256: exigir(h == sha256, f"SHA-256 distinto: {nombre}; no aceptar otra versión en silencio.")
    FUENTES[nombre] = dict(url=url, sha256=h, bytes=ruta.stat().st_size)
    return ruta

def insertar(destino, clave, valor, contexto):
    clave = str(clave)
    exigir(clave not in destino, f"ID duplicado en {contexto}: {clave}")
    destino[clave] = valor

def armar_tarea(nombre, docs, queries, qrels, grupos, fuente, self_filter=False):
    exigir(docs and queries and qrels, f"Tarea vacía: {nombre}")
    for q, rs in qrels.items():
        exigir(q in queries, f"Query de qrels ausente: {nombre}/{q}")
        exigir(any(g > 0 for g in rs.values()), f"Query sin positivo publicado: {nombre}/{q}")
        for d, g in rs.items():
            exigir(d in docs, f"Documento anotado ausente: {nombre}/{d}")
            exigir(isinstance(g, int) and g >= 0, "Grado inválido")
            if self_filter: exigir(not (d == q and g > 0), "Un positivo coincide con el query ID.")
    queries = {q: queries[q] for q in sorted(qrels)}
    docs = dict(sorted(docs.items()))
    exigir(all(isinstance(t, str) and t.strip() for t in docs.values()), "Documento vacío/no textual")
    exigir(all(isinstance(t, str) and t.strip() for t in queries.values()), "Query vacío/no textual")
    tarea = dict(nombre=nombre, docs=docs, queries=queries, qrels=qrels,
                 grupos={q: grupos.get(q, q) for q in queries}, fuente=fuente, self_filter=self_filter)
    tarea["firma"] = firma(tarea)
    TAREAS[nombre] = tarea
    return tarea

def grupos_scifact(qrels):
    parent = {q: q for q in qrels}
    def raiz(q):
        while parent[q] != q:
            parent[q] = parent[parent[q]]; q = parent[q]
        return q
    vistos = {}
    for q, rs in qrels.items():
        for d, g in rs.items():
            if g > 0:
                if d in vistos: parent[raiz(q)] = raiz(vistos[d])
                else: vistos[d] = q
    return {q: raiz(q) for q in qrels}

BEIR = {"scifact": "5f7d1de60b170fc8027bb7898e2efca1",
        "scidocs": "38121350fc3a4d2f48850f6aff52e4a9",
        "nfcorpus": "a89dba18a62ef92f7d323ec890a0d38d"}
for nombre, md5 in BEIR.items():
    ruta = descargar(f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{nombre}.zip",
                     nombre + ".zip", md5=md5)
    with zipfile.ZipFile(ruta) as z:
        def miembro(final):
            encontrados = [n for n in z.namelist() if n.endswith("/" + final) or n == final]
            exigir(len(encontrados) == 1, f"Archivo ambiguo/ausente: {final}")
            return encontrados[0]
        docs, queries, qrels = {}, {}, {}
        with z.open(miembro("corpus.jsonl")) as f:
            for linea in f:
                row = json.loads(linea)
                texto = (str(row.get("title") or "") + "\n" + row["text"]).strip()
                insertar(docs, row["_id"], texto, nombre + " corpus")
        with z.open(miembro("queries.jsonl")) as f:
            for linea in f:
                row = json.loads(linea); insertar(queries, row["_id"], row["text"], nombre + " queries")
        with z.open(miembro("qrels/test.tsv")) as f:
            tabla = pd.read_csv(f, sep="\t", dtype=str, keep_default_na=False)
        exigir({"query-id", "corpus-id", "score"} <= set(tabla.columns), "Esquema qrels BEIR inesperado")
        for q, d, g in tabla[["query-id", "corpus-id", "score"]].itertuples(index=False, name=None):
            insertar(qrels.setdefault(q, {}), d, int(g), "qrels " + q)
    grupos = grupos_scifact(qrels) if nombre == "scifact" else {}
    armar_tarea(nombre, docs, queries, qrels, grupos, dict(tipo="BEIR", split="test", **FUENTES[nombre + ".zip"]), True)
    print(nombre, "| documentos:", len(docs), "| consultas evaluadas:", len(qrels))

repo_miracl = "mteb/MIRACLRetrievalHardNegatives"
pin = CACHE / "miracl_revision.json"
if pin.exists():
    revision_miracl = json.loads(pin.read_text(encoding="utf-8"))["revision"]
else:
    revision_miracl = HfApi().dataset_info(repo_miracl).sha
    exigir(bool(re.fullmatch(r"[0-9a-f]{40}", revision_miracl or "")), "Revisión MIRACL inválida")
    guardar(pin, dict(repo=repo_miracl, revision=revision_miracl, fijado_utc=ahora()))
lista_pin = CACHE / "miracl_archivos.json"
if lista_pin.exists():
    lista = json.loads(lista_pin.read_text(encoding="utf-8"))
else:
    lista = HfApi().list_repo_files(repo_miracl, repo_type="dataset", revision=revision_miracl)
    guardar(lista_pin, lista)

miracl_fuentes = []
def parquet_miracl(tipo):
    archivos = sorted(n for n in lista if n.startswith(f"es-{tipo}/") and n.endswith(".parquet")
                      and (tipo == "corpus" or Path(n).name.startswith("dev-")))
    exigir(archivos, f"No se encontró el split MIRACL es-{tipo} esperado.")
    marcos = []
    for n in archivos:
        ruta = Path(hf_hub_download(repo_id=repo_miracl, repo_type="dataset", filename=n, revision=revision_miracl))
        miracl_fuentes.append(dict(archivo=n, sha256=hash_archivo(ruta)))
        marcos.append(pd.read_parquet(ruta))
    return pd.concat(marcos, ignore_index=True)

def columna(df, candidatos, opcional=False):
    c = next((x for x in candidatos if x in df.columns), None)
    if c is None and not opcional: raise ValueError(f"Esquema inesperado: {df.columns.tolist()}; esperaba {candidatos}")
    return c

f = parquet_miracl("corpus")
ci = columna(f, ["id", "_id", "docid", "doc_id", "corpus_id", "corpus-id"])
ct = columna(f, ["text", "contents", "passage"])
ch = columna(f, ["title"], True)
docs = {}
for row in f[[ci, ct] + ([ch] if ch else [])].itertuples(index=False, name=None):
    exigir(isinstance(row[1], str), "Texto MIRACL inválido")
    titulo = row[2] if ch and isinstance(row[2], str) else ""
    insertar(docs, row[0], (titulo + "\n" + row[1]).strip(), "MIRACL corpus")
del f
f = parquet_miracl("queries")
ci = columna(f, ["id", "_id", "query_id", "query-id"])
ct = columna(f, ["text", "query", "question"])
queries = {}
for q, t in f[[ci, ct]].itertuples(index=False, name=None): insertar(queries, q, t, "MIRACL queries")
f = parquet_miracl("qrels")
cq = columna(f, ["query-id", "query_id", "qid"])
cd = columna(f, ["corpus-id", "corpus_id", "doc_id", "docid"])
cg = columna(f, ["score", "relevance", "relevancy", "label"])
qrels = {}
for q, d, g in f[[cq, cd, cg]].itertuples(index=False, name=None):
    exigir(float(g).is_integer(), "Grado MIRACL no entero")
    insertar(qrels.setdefault(str(q), {}), d, int(g), "MIRACL qrels")
# Los juicios MIRACL HN pueden incluir IDs fuera del corpus de candidatos.
# No inventar documentos ni tratar positivos ausentes como no pertinentes.
originales = len(qrels)
originales_juicios = sum(len(rs) for rs in qrels.values())
positivos_fuera = [(q, d) for q, rs in qrels.items() for d, g in rs.items()
                   if g > 0 and d not in docs]
negativos_fuera = sum(g == 0 and d not in docs for rs in qrels.values()
                     for d, g in rs.items())
consultas_excluidas = sorted({q for q, _ in positivos_fuera})
qrels_compatibles = {
    q: {d: g for d, g in rs.items() if d in docs}
    for q, rs in qrels.items() if q not in consultas_excluidas
}
qrels_compatibles = {
    q: rs for q, rs in qrels_compatibles.items() if any(g > 0 for g in rs.values())
}
resumen_miracl = dict(
    corpus_candidatos=len(docs), consultas_originales=originales,
    consultas_evaluadas=len(qrels_compatibles),
    consultas_excluidas_por_positivos_ausentes=len(consultas_excluidas),
    positivos_ausentes_en_corpus=len(positivos_fuera),
    juicios_no_pertinentes_fuera_de_corpus=negativos_fuera,
    juicios_originales=originales_juicios,
    juicios_evaluados=sum(len(rs) for rs in qrels_compatibles.values()),
    politica="Excluir toda consulta si falta al menos un positivo publicado; "
             "ignorar únicamente juicios grado 0 de documentos fuera del corpus"
)
exigir(len(qrels_compatibles) > 0, "MIRACL HN no tiene consultas completas; no continuar sin corpus adecuado.")
armar_tarea("miracl_es_hn", docs, queries, qrels_compatibles, {},
            dict(repo=repo_miracl, revision=revision_miracl, split="dev",
                 archivos=miracl_fuentes, control_cobertura=resumen_miracl,
                 alcance="Corpus de candidatos HN; evaluación condicionada a positivos íntegros por consulta; "
                         "NO corpus MIRACL original completo ni test oficial sin filtrado"))
print("miracl_es_hn | documentos:", len(docs), "| consultas evaluadas:", len(qrels_compatibles))
print("MIRACL: consultas excluidas por positivos ausentes:", len(consultas_excluidas),
      "| positivos ausentes:", len(positivos_fuera),
      "| juicios grado 0 ausentes:", negativos_fuera)
if len(consultas_excluidas) > 0:
    print("Advertencia: resultados MIRACL HN condicionados a consultas con todos los positivos presentes.")
del f

SHA_XQUAD = {"en": "e4c57d1c9143aaa1c5d265ba5987a65f4e69528d2a98f29d6e75019b10344f29",
             "es": "dcbae93ec3a9f4b9e78fd834a171d6f96c1a875e10e15b7530b7e4ef4971e37e"}
XQ = {}
for idioma in ("en", "es"):
    nombre = f"xquad.{idioma}.json"
    ruta = descargar(f"https://raw.githubusercontent.com/google-deepmind/xquad/master/{nombre}", nombre,
                     sha256=SHA_XQUAD[idioma])
    x = json.loads(ruta.read_text(encoding="utf-8"))
    docs, queries, qrels, grupos = {}, {}, {}, {}
    for articulo in x["data"]:
        for parrafo in articulo["paragraphs"]:
            did = "p_" + firma(sorted(q["id"] for q in parrafo["qas"]))[:24]
            insertar(docs, did, parrafo["context"], nombre)
            for q in parrafo["qas"]:
                for a in q["answers"]:
                    inicio = a["answer_start"]
                    exigir(parrafo["context"][inicio:inicio + len(a["text"])] == a["text"], "Respuesta XQuAD no literal")
                insertar(queries, q["id"], q["question"], nombre)
                qrels[q["id"]] = {did: 1}; grupos[q["id"]] = did
    exigir(len(docs) == 240 and len(queries) == 1190, "El XQuAD descargado no tiene el tamaño publicado.")
    XQ[idioma] = dict(docs=docs, queries=queries, qrels=qrels, grupos=grupos)
exigir(XQ["es"]["qrels"] == XQ["en"]["qrels"], "XQuAD ES/EN no está alineado por IDs.")
for qlang, dlang in [("es", "en"), ("en", "en"), ("es", "es")]:
    armar_tarea(f"xquad_{qlang}_{dlang}", XQ[dlang]["docs"], XQ[qlang]["queries"], XQ[qlang]["qrels"],
                XQ[qlang]["grupos"], dict(tipo="XQuAD adaptado a recuperación del pasaje de origen",
                  idiomas=[qlang, dlang], archivos=[FUENTES["xquad.en.json"], FUENTES["xquad.es.json"]]))

resumen = {n: dict(firma=t["firma"], documentos=len(t["docs"]), consultas=len(t["queries"]),
           bloques_inferencia=len(set(t["grupos"].values())), fuente=t["fuente"],
           peso_principal=PESOS.get(n, 0.0)) for n, t in TAREAS.items()}
if R["datasets"]: exigir(R["datasets"] == resumen, "Cambió el contenido de los datos externos.")
R["datasets"] = resumen
R["estado"] = "datos_validados"
guardar(SALIDA, R)
display(pd.DataFrame(resumen).T[["documentos", "consultas", "bloques_inferencia", "peso_principal"]])

## 2. Métricas y controles unitarios

Se reportan nDCG, Precision, Recall y MRR en 5/10/20/100; se mantienen los grados publicados. **El nDCG principal usa ganancia lineal**, como `trec_eval`, no la ganancia exponencial de la evaluación interna anterior. Los scores de ambas pruebas no deben mezclarse.

`Judged@k` informa cuántos resultados tienen un juicio publicado; los demás no reciben etiquetas nuevas, aunque aportan cero al cálculo convencional del benchmark. Recall se refiere al total de positivos publicados. No se mide abstención: estas tareas no acreditan consultas sin respuesta.

In [ ]:
def calcular_metricas(ranking, qrels, cortes=KS):
    total = sum(g > 0 for g in qrels.values())
    exigir(total > 0 and len(ranking) == len(set(ranking)), "Qrels/ranking inválido")
    out = {}
    for k in cortes:
        encontrados = ranking[:k]
        grados = np.asarray([qrels.get(d, 0) for d in encontrados], dtype=float)
        ideal = np.asarray(sorted(qrels.values(), reverse=True)[:k], dtype=float)
        dcg = float(np.sum(grados / np.log2(np.arange(len(grados)) + 2)))
        idcg = float(np.sum(ideal / np.log2(np.arange(len(ideal)) + 2)))
        posiciones = np.flatnonzero(grados > 0)
        out.update({f"nDCG@{k}": dcg / idcg, f"Precision@{k}": len(posiciones) / k,
                    f"Recall@{k}": len(posiciones) / total,
                    f"MRR@{k}": 1.0 / (int(posiciones[0]) + 1) if len(posiciones) else 0.0,
                    f"Judged@{k}": sum(d in qrels for d in encontrados) / k})
    return out

def buscar_exacto(Q, X, dids, qids, excluir_mismo_id=False, top=100):
    exigir(dids == sorted(dids) and len(set(dids)) == len(dids), "IDs documentales desordenados")
    indices = {d: i for i, d in enumerate(dids)}
    rr, ss = [], []
    for inicio in range(0, len(qids), 16):
        scores = np.asarray(Q[inicio:inicio+16] @ X.T, dtype=np.float32)
        exigir(np.isfinite(scores).all(), "Score no finito")
        for j, fila in enumerate(scores):
            qid = qids[inicio + j]
            if excluir_mismo_id and qid in indices: fila[indices[qid]] = -np.inf
            orden = np.argsort(-fila, kind="stable")
            orden = orden[np.isfinite(fila[orden])][:top]
            rr.append(orden.astype(np.int32)); ss.append(fila[orden])
    return np.stack(rr), np.stack(ss)

m = calcular_metricas(["x", "b", "a"], {"a": 2, "b": 1}, [1, 3])
assert m["Precision@1"] == 0 and m["MRR@3"] == .5 and m["Recall@3"] == 1
assert abs(m["nDCG@3"] - ((1 / np.log2(3) + 2 / np.log2(4)) / (2 + 1 / np.log2(3)))) < 1e-12
assert calcular_metricas(["a"], {"a": 1}, [5])["Precision@5"] == .2
r0, _ = buscar_exacto(np.array([[1., 0.]], np.float32), np.array([[1., 0.], [1., 0.], [0., 1.]], np.float32),
                     ["a", "b", "c"], ["a"], True, 2)
assert r0.tolist() == [[1, 2]]
print("Controles de métricas, denominadores, empates y exclusión de auto-recuperación: correctos.")

## 3. Codificación, formatos y reanudación

Se respeta la revisión y el formato propio de cada modelo. No se cambian pesos, pooling ni precisión para favorecer a un finalista. No se aceptan truncamientos silenciosos: un texto que exceda el límite nativo detiene la tarea.

Los checkpoints se escriben por bloques, con firma de entradas, entorno y bytes de vectores. Un fallo de memoria puede reducir únicamente el tamaño del lote, dejando registro; no acorta texto ni cambia dtype. La búsqueda se hace sobre **todos** los documentos de cada corpus declarado. No se utilizan los modelos para generar o juzgar sus propias respuestas correctas.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

exigir(torch.cuda.is_available(), "CUDA no está disponible en este kernel.")
exigir(torch.cuda.is_bf16_supported(), "Esta prueba requiere BF16 nativo.")
torch.manual_seed(SEMILLA)
np.random.seed(SEMILLA)
torch.set_float32_matmul_precision("highest")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
ENTORNO = {n: importlib.metadata.version(n) for n in
           ["numpy", "pandas", "torch", "sentence-transformers", "transformers", "huggingface_hub", "pyarrow"]}
ENTORNO.update(python=sys.version.split()[0], sistema=platform.platform(),
               gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda)
print("GPU:", ENTORNO["gpu"], "| libre:", round(torch.cuda.mem_get_info()[0] / 2**30, 2), "GiB")

def cargar_modelo(cfg):
    kw = dict(dtype=getattr(torch, cfg["dtype"]), attn_implementation="sdpa", use_safetensors=True)
    otros = {}
    if cfg["remote"]:
        kw["code_revision"] = cfg["revision"]
        otros["config_kwargs"] = dict(code_revision=cfg["revision"])
    m = SentenceTransformer(cfg["repo"], revision=cfg["revision"], device="cuda",
                            trust_remote_code=cfg["remote"], model_kwargs=kw, **otros)
    m.eval()
    exigir(m.get_sentence_embedding_dimension() == cfg["dim"], "Dimensión inesperada")
    if cfg["prompt_name"]:
        exigir(m.prompts.get(cfg["prompt_name"]) == cfg["query"], "Prefijo de consulta distinto del fijado")
    if cfg["document"]: exigir(m.prompts.get("document") == cfg["document"], "Prefijo documental inesperado")
    if cfg["remote"]: exigir(not any(m.prompts.values()), "PPLX no debe añadir instrucciones")
    m.max_seq_length = cfg["max_tokens"]
    dtypes = sorted({str(p.dtype) for p in m.parameters()})
    exigir(dtypes == [str(getattr(torch, cfg["dtype"]))], f"Precisión inesperada: {dtypes}")
    return m, dict(dtypes=dtypes, prompts=dict(m.prompts), max_seq_length=m.max_seq_length,
                   dimension=m.get_sentence_embedding_dimension(),
                   parametros=sum(p.numel() for p in m.parameters()),
                   arquitectura=m[0].auto_model.config.model_type)

def normalizar(x):
    x = np.asarray(x, dtype=np.float32).copy(order="C")
    normas = np.linalg.norm(x, axis=1, keepdims=True)
    exigir(x.ndim == 2 and np.isfinite(x).all() and np.all(normas > 0), "Embedding inválido")
    x /= normas
    return x

def encode_controlado(m, textos, prefijo, cfg):
    longitudes = [len(x) for x in m.tokenizer([prefijo + t for t in textos],
        truncation=False, padding=False, add_special_tokens=True)["input_ids"]]
    exigir(max(longitudes) <= LIMITE_FRAGMENTO, "Error de fragmentación: entrada > 1024 tokens")
    batch = min(cfg["batch"], len(textos)); intentos = []
    while True:
        try:
            torch.cuda.synchronize(); t0 = perf_counter()
            with torch.inference_mode():
                v = m.encode(textos, prompt=prefijo, batch_size=batch,
                    normalize_embeddings=False, precision="float32",
                    convert_to_numpy=True, show_progress_bar=False)
            torch.cuda.synchronize()
            elapsed = perf_counter() - t0
            v = normalizar(v)
            exigir(v.shape == (len(textos), cfg["dim"]), "Dimensión de vectores inesperada")
            return v, dict(segundos=elapsed, batch_real=batch, reintentos_oom=intentos)
        except torch.cuda.OutOfMemoryError:
            if batch == 1: raise
            intentos.append(batch); batch = max(1, batch // 2)
            gc.collect(); torch.cuda.empty_cache()

def tokenizar_entrada(m, texto, prefijo):
    tk = m.tokenizer
    completo = tk(prefijo + texto, truncation=False, padding=False,
                  add_special_tokens=True)["input_ids"]
    return len(completo)

def segmentar(m, texto, prefijo):
    """Ventanas de tokens que cubren el texto íntegro; ninguna pasada excede el límite."""
    tk = m.tokenizer
    ids = tk(texto, truncation=False, padding=False, add_special_tokens=False)["input_ids"]
    # Margen para tokens especiales y el prefijo; al decodificar se verifica de nuevo.
    extra = len(tk(prefijo, truncation=False, add_special_tokens=True)["input_ids"])
    ancho = LIMITE_FRAGMENTO - extra - 16
    exigir(ancho > SOLAPAMIENTO_TOKENS + 64, "Prefijo excesivo")
    if tokenizar_entrada(m, texto, prefijo) <= LIMITE_FRAGMENTO:
        return [(texto, len(ids))]
    partes=[]; inicio=0
    while inicio < len(ids):
        fin = min(inicio+ancho, len(ids))
        # Si la retokenización cambia el tamaño por el corte, reducir el tramo.
        while True:
            trozo=tk.decode(ids[inicio:fin], skip_special_tokens=True,
                            clean_up_tokenization_spaces=False)
            n=tokenizar_entrada(m, trozo, prefijo)
            if n <= LIMITE_FRAGMENTO: break
            fin -= max(8, int((fin-inicio)*0.05))
            exigir(fin>inicio, "No se pudo adaptar el tamaño de un fragmento")
        nuevos = fin-inicio if not partes else fin-inicio-SOLAPAMIENTO_TOKENS
        exigir(nuevos>0 and trozo.strip(), "Fragmento inválido")
        partes.append((trozo, nuevos))
        if fin == len(ids): break
        inicio = fin-SOLAPAMIENTO_TOKENS
    exigir(len(partes)>=2, "No se fragmentó texto largo")
    return partes

def checkpoint_anterior(cfg, clave, ids, textos, tipo, entorno):
    """Abrir V1 solo para lectura tras validar condición y bloques SHA256."""
    if ANTERIOR is None or clave not in ANTERIOR.get("modelos", {}):
        return None, 0
    viejo_firma = ANTERIOR["firma_protocolo"]
    condicion = dict(modelo=cfg, tipo=tipo, prefijo=cfg[tipo],
                     entradas=firma(list(zip(ids, textos))), entorno=entorno, protocolo=viejo_firma)
    h = firma(condicion)
    carpeta = CACHE / "vectores" / clave
    p = carpeta / (h + ".npy"); meta = carpeta / (h + ".json")
    if not p.exists() or not meta.exists(): return None, 0
    st = json.loads(meta.read_text(encoding="utf-8"))
    exigir(st["condicion"]==condicion, "Condición de caché V1 alterada")
    V = np.load(p, mmap_mode="r", allow_pickle=False)
    exigir(V.shape == (len(ids),cfg["dim"]) and V.dtype==np.float32, "Vectores V1 inválidos")
    exigir(st["completados"] >= 0 and st["completados"]<=len(ids), "Contador V1 inválido")
    pos=0
    for b in st["bloques"]:
        exigir(b["inicio"]==pos and b["fin"]<=st["completados"], "Bloques V1 no consecutivos")
        exigir(hashlib.sha256(V[b["inicio"]:b["fin"]].tobytes()).hexdigest()==b["sha256"],
               "Vectores V1 modificados; no reusar")
        pos=b["fin"]
    exigir(pos==st["completados"], "Punto de control V1 incompleto")
    return V, pos

def vectores_cache(m, clave, cfg, ids, textos, tipo):
    condicion = dict(modelo=cfg, tipo=tipo, prefijo=cfg[tipo],
                     entradas=firma(list(zip(ids, textos))), entorno=ENTORNO, protocolo=FIRMA_PROTOCOLO)
    h = firma(condicion)
    carpeta = CACHE / "vectores" / clave
    carpeta.mkdir(parents=True, exist_ok=True)
    p = carpeta / (h + ".npy"); estado = carpeta / (h + ".json")
    forma=(len(ids),cfg["dim"])
    if estado.exists():
        st=json.loads(estado.read_text(encoding="utf-8"))
        exigir(st["condicion"]==condicion and p.is_file(), "Checkpoint V2 incompatible/incompleto")
        V=np.lib.format.open_memmap(p,mode="r+")
        exigir(V.shape==forma and V.dtype==np.float32, "Matriz V2 incompatible")
        pos=0
        for b in st["bloques"]:
            exigir(b["inicio"]==pos and b["fin"]<=st["completados"], "Bloques V2 no consecutivos")
            exigir(hashlib.sha256(V[b["inicio"]:b["fin"]].tobytes()).hexdigest()==b["sha256"],
                   "Bloque V2 modificado; detener para proteger datos")
            pos=b["fin"]
        exigir(pos==st["completados"], "Contador V2 incoherente")
    else:
        V=np.lib.format.open_memmap(p,mode="w+",dtype=np.float32,shape=forma)
        st=dict(condicion=condicion,completados=0,bloques=[],max_tokens_observados=0)
        guardar(estado,st)
    V1, previos = checkpoint_anterior(cfg, clave, ids, textos, tipo, ENTORNO)
    if previos: print(clave, tipo, "checkpoint V1 validado:", previos, flush=True)
    for i in range(st["completados"],len(ids),TAM_BLOQUE):
        fin=min(i+TAM_BLOQUE,len(ids)); bloques=textos[i:fin]
        vectores=np.empty((len(bloques),cfg["dim"]),np.float32)
        para_codificar=[]; pesos=[]; indices=[]; maximo=0; reutilizados=0; largos=0
        for j,texto in enumerate(bloques):
            largo=tokenizar_entrada(m,texto,cfg[tipo]);maximo=max(maximo,largo)
            if V1 is not None and i+j<previos and largo<=LIMITE_FRAGMENTO:
                vectores[j]=V1[i+j];reutilizados+=1;continue
            partes=segmentar(m,texto,cfg[tipo])
            if len(partes)>1: largos+=1
            for trozo,peso in partes:
                para_codificar.append(trozo);pesos.append(peso);indices.append(j)
        segundos=0.0; reintentos=[]; batch_real=cfg["batch"]
        if para_codificar:
            # Particionar las llamadas evita listas enormes cuando aparece un documento extenso.
            w=np.zeros_like(vectores); tot=np.zeros((len(bloques),),np.float64)
            for ini in range(0,len(para_codificar),32):
                final=min(ini+32,len(para_codificar))
                z,info=encode_controlado(m,para_codificar[ini:final],cfg[tipo],cfg)
                segundos+=info["segundos"];reintentos+=info["reintentos_oom"]
                batch_real=min(batch_real,info["batch_real"])
                for k,emb in enumerate(z,ini):
                    j=indices[k];peso=pesos[k];w[j]+=emb*peso;tot[j]+=peso
            for j in set(indices):
                exigir(tot[j]>0, "Documento sin fragmentos")
                vectores[j]=normalizar(w[j:j+1])[0]
        exigir(np.isfinite(vectores).all(), "Vectores no finitos")
        V[i:fin]=vectores;V.flush()
        st["completados"]=fin
        st["max_tokens_observados"]=max(st["max_tokens_observados"],maximo)
        st["bloques"].append(dict(inicio=i,fin=fin,
            sha256=hashlib.sha256(vectores.tobytes()).hexdigest(),segundos=segundos,
            batch_real=batch_real,reintentos_oom=reintentos,
            reusados_v1=reutilizados,documentos_fragmentados=largos,
            fragmentos_calculados=len(para_codificar)))
        guardar(estado,st)
        if fin==len(ids) or fin%256==0:
            print(clave,tipo,fin,"/",len(ids),"| reciclados",reutilizados,
                  "| fragmentados",largos,flush=True)
    resumen=dict(firma=h,archivo_cache=str(p.relative_to(BASE)),sha256=hash_archivo(p),
        vectores=len(ids),dimension=cfg["dim"],max_tokens=st["max_tokens_observados"],truncados=0,
        documentos_fragmentados=sum(b["documentos_fragmentados"] for b in st["bloques"]),
        vectores_reusados_v1=sum(b["reusados_v1"] for b in st["bloques"]),
        fragmentos_calculados=sum(b["fragmentos_calculados"] for b in st["bloques"]),
        segundos_codificacion_acumulados=sum(b["segundos"] for b in st["bloques"]),
        bloques_con_reintentos_oom=sum(bool(b["reintentos_oom"]) for b in st["bloques"]))
    return V,resumen


## 4. Ejecutar los cinco modelos

Se completan todas las tareas por modelo y se libera la GPU antes del siguiente. Las tareas terminadas se reutilizan. El resultado permanece **incompleto** si falta un modelo o un dataset. Las mediciones de esta libreta son tiempos de etapa y consumo PyTorch; no se presentan como latencia interactiva comparable entre tamaños de lote.
**Importante:** los resultados y caches V1 no se sobrescriben; la versión 2 usa segmentación explícita para documentos extensos y conserva todos los tokens.


In [ ]:
ORDEN_TAREAS = ["scifact", "scidocs", "nfcorpus", "miracl_es_hn", "xquad_es_en", "xquad_en_en", "xquad_es_es"]
for clave, cfg in MODELOS.items():
    registro = R["modelos"].setdefault(clave, dict(configuracion=cfg, tareas={}))
    exigir(registro["configuracion"] == cfg, "Cambió la configuración del modelo")
    if registro.get("entorno"):
        exigir(registro["entorno"] == ENTORNO, "Cambió el entorno; no mezclar ejecuciones en la misma carpeta")
    registro["entorno"] = ENTORNO
    pendientes = [n for n in ORDEN_TAREAS if n not in registro["tareas"]]
    if not pendientes:
        print(clave, "ya completo"); continue
    m = None; X = Q = None
    try:
        gc.collect(); torch.cuda.empty_cache()
        exigir(torch.cuda.mem_get_info()[0] > (9 if cfg["dim"] == 2560 else 5) * 2**30,
               "GPU ocupada: cierra otros kernels/modelos antes de continuar.")
        torch.cuda.reset_peak_memory_stats()
        t0 = perf_counter(); m, runtime = cargar_modelo(cfg)
        registro.setdefault("cargas", []).append(dict(fecha_utc=ahora(), segundos=perf_counter()-t0, runtime=runtime))
        encode_controlado(m, ["How can relevant research evidence be retrieved?"], cfg["query"], cfg)
        for nombre in pendientes:
            t = TAREAS[nombre]
            print("\n", clave, "|", nombre, flush=True)
            dids, qids = list(t["docs"]), list(t["queries"])
            t0 = perf_counter()
            X, meta_x = vectores_cache(m, clave, cfg, dids, list(t["docs"].values()), "document")
            Q, meta_q = vectores_cache(m, clave, cfg, qids, list(t["queries"].values()), "query")
            tbus = perf_counter()
            indices, scores = buscar_exacto(Q, X, dids, qids, t["self_filter"], max(KS))
            busqueda_s = perf_counter() - tbus
            filas = []
            for q, ranks in zip(qids, indices):
                ranking = [dids[int(i)] for i in ranks]
                filas.append(dict(query_id=q, bloque=t["grupos"][q],
                                  **calcular_metricas(ranking, t["qrels"][q])))
            marco = pd.DataFrame(filas)
            metricas = [c for c in marco if "@" in c]
            medias = {c: float(marco[c].mean()) for c in metricas}
            ruta_rank = CACHE / "rankings" / f"{clave}__{nombre}.npz"
            ruta_rank.parent.mkdir(exist_ok=True)
            with ruta_rank.open("wb") as f:
                np.savez_compressed(f, indices=indices, scores=scores, query_ids=np.asarray(qids),
                                    doc_ids=np.asarray(dids))
            datos = dict(tarea=nombre, firma_dataset=t["firma"], firma_protocolo=FIRMA_PROTOCOLO,
                corpus=meta_x, queries=meta_q, promedios=medias, por_consulta=filas,
                archivo_rankings=str(ruta_rank.relative_to(BASE)), sha256_rankings=hash_archivo(ruta_rank),
                recursos=dict(segundos_busqueda_exhaustiva=busqueda_s,
                    segundos_etapa_actual=perf_counter()-t0,
                    pico_pytorch_asignado_bytes=torch.cuda.max_memory_allocated(),
                    pico_pytorch_reservado_bytes=torch.cuda.max_memory_reserved(),
                    alcance="pico acumulado del proceso PyTorch desde carga; no VRAM total; no latencia por consulta"),
                fecha_utc=ahora())
            registro["tareas"][nombre] = dict(datos=datos, firma=firma(datos))
            R["estado"] = "en_ejecucion"; guardar(SALIDA, R)
            print("nDCG@10 =", round(medias["nDCG@10"], 5), "| guardado", flush=True)
            del X, Q; X = Q = None; gc.collect()
    except Exception as error:
        R["estado"] = "incompleto"
        R["incidencias"].append(dict(modelo=clave, fecha_utc=ahora(), tipo=type(error).__name__, mensaje=str(error)))
        guardar(SALIDA, R)
        raise
    finally:
        X = Q = None
        if m is not None: del m
        gc.collect(); torch.cuda.empty_cache()
exigir(all(set(r["tareas"]) == set(ORDEN_TAREAS) for r in R["modelos"].values()) and
       set(R["modelos"]) == set(MODELOS), "No se completaron los cinco modelos")
R["estado"] = "recuperacion_completa"; guardar(SALIDA, R)
print("Recuperación completa. Sigue la comparación pareada, sin nuevas inferencias.")

## 5. Comparar sin forzar un ganador

**Regla fijada antes de esta ejecución:** promedio de nDCG@10 con peso 20% para cada una de las cinco tareas principales. Es una decisión de diseño, no la distribución real de consultas de tu tesis. Se muestran además cada tarea y las diferencias de Recall; no se mezcla calidad con velocidad.

El bootstrap es pareado: se remuestrean los mismos bloques para todos los modelos. SciFact agrupa consultas con evidencia positiva compartida y XQuAD agrupa las preguntas por su pasaje original. Para las otras tareas se usa query ID; la dependencia residual entre temas queda como limitación. Los CI se ajustan por los diez pares con Bonferroni. No son garantías universales ni eliminan errores de anotación o exposición previa de los modelos a los datos.

Se declara una separación amplia en **este benchmark** únicamente si el líder supera a cada competidor por más de 0.01 en el límite inferior simultáneo y no pierde más de 0.02 en ninguna tarea principal. El margen 0.01 y el control 0.02 son criterios operativos predefinidos, no constantes universales. De lo contrario se devuelve un conjunto no descartado. **No significancia no significa equivalencia.**

In [ ]:
ids_modelos = list(MODELOS)
exigir(set(R["modelos"]) == set(ids_modelos), "Faltan modelos")
for s in ids_modelos:
    exigir(set(R["modelos"][s]["tareas"]) == set(ORDEN_TAREAS), f"Faltan tareas: {s}")
    for n in ORDEN_TAREAS:
        run = R["modelos"][s]["tareas"][n]
        exigir(firma(run["datos"]) == run["firma"], "Firma de tarea incorrecta")
        exigir(run["datos"]["firma_dataset"] == TAREAS[n]["firma"], "Dataset incompatible")

filas_resumen = [dict(modelo=s, tarea=n, **R["modelos"][s]["tareas"][n]["datos"]["promedios"])
                 for s in ids_modelos for n in ORDEN_TAREAS]
TABLA = pd.DataFrame(filas_resumen)
pivot = TABLA.pivot(index="modelo", columns="tarea", values="nDCG@10").loc[ids_modelos]
puntos = sum(PESOS[n] * pivot[n].to_numpy() for n in PESOS)

rng = np.random.default_rng(SEMILLA)
boot = np.zeros((BOOTSTRAPS, len(ids_modelos)), dtype=np.float64)
info_bloques = {}
for nombre, peso in PESOS.items():
    marcos = [pd.DataFrame(R["modelos"][s]["tareas"][nombre]["datos"]["por_consulta"]).set_index("query_id")
              for s in ids_modelos]
    qids = sorted(marcos[0].index)
    exigir(all(set(d.index) == set(qids) for d in marcos), "Queries desalineados entre modelos")
    val = np.column_stack([d.loc[qids, "nDCG@10"].to_numpy() for d in marcos])
    grupos = [TAREAS[nombre]["grupos"][q] for q in qids]
    unicos = sorted(set(grupos)); idx = {g: i for i, g in enumerate(unicos)}
    sums = np.zeros((len(unicos), len(ids_modelos))); counts = np.zeros(len(unicos))
    for fila, g in zip(val, grupos): sums[idx[g]] += fila; counts[idx[g]] += 1
    exigir(len(unicos) > 1, f"Muy pocos bloques para inferencia: {nombre}")
    info_bloques[nombre] = dict(consultas=len(qids), bloques=len(unicos))
    for inicio in range(0, BOOTSTRAPS, 128):
        fin = min(inicio+128, BOOTSTRAPS)
        sample = rng.integers(0, len(unicos), size=(fin-inicio, len(unicos)))
        boot[inicio:fin] += peso * (sums[sample].sum(axis=1) / counts[sample].sum(axis=1)[:, None])

pares = list(combinations(range(len(ids_modelos)), 2))
cola = ALFA_FAMILIA / (2 * len(pares))
comparaciones, intervalos = [], {}
for i, j in pares:
    delta = float(puntos[i] - puntos[j])
    lo, hi = map(float, np.quantile(boot[:, i] - boot[:, j], [cola, 1-cola]))
    intervalos[i, j] = (lo, hi); intervalos[j, i] = (-hi, -lo)
    relacion = ("a_superior_margen" if lo > MARGEN_PRACTICO else
                "b_superior_margen" if hi < -MARGEN_PRACTICO else
                "equivalencia_practica_en_margen" if lo >= -MARGEN_PRACTICO and hi <= MARGEN_PRACTICO else
                "diferencia_no_resuelta")
    comparaciones.append(dict(a=ids_modelos[i], b=ids_modelos[j], delta_ndcg10=delta,
                              ci_simultaneo_inf=lo, ci_simultaneo_sup=hi, diagnostico=relacion))

lider_i = int(np.argmax(puntos)); lider = ids_modelos[lider_i]
no_descartados = [ids_modelos[i] for i in range(len(ids_modelos))
                  if not any(intervalos[j, i][0] > MARGEN_PRACTICO for j in range(len(ids_modelos)) if j != i)]
separacion = all(intervalos[lider_i, j][0] > MARGEN_PRACTICO for j in range(len(ids_modelos)) if j != lider_i)
control_tareas = all(pivot.loc[lider, n] >= pivot.loc[s, n] - .02
                     for n in PESOS for s in ids_modelos if s != lider)
resumen_modelos = [dict(modelo=s, ndcg10_macro_ponderado=float(puntos[i]),
                        ndcg10_por_tarea={n: float(pivot.loc[s,n]) for n in ORDEN_TAREAS})
                   for i, s in enumerate(ids_modelos)]
resumen_modelos.sort(key=lambda x: -x["ndcg10_macro_ponderado"])
R["comparacion"] = dict(resumen=resumen_modelos, pares=comparaciones, bloques=info_bloques,
    nivel_ci_individual=1-ALFA_FAMILIA/len(pares), metodo="bootstrap percentil pareado por bloques + Bonferroni",
    lider_puntual=lider, no_descartados=no_descartados,
    superioridad_amplia_en_este_benchmark=bool(separacion and control_tareas),
    control_perdida_maxima_002_por_tarea=bool(control_tareas),
    seleccion_definitiva_para_tesis=False,
    conclusion=("Separación a favor del líder bajo los criterios del benchmark externo." if separacion and control_tareas else
                "No forzar un ganador: conservar los no descartados y examinar el perfil por tarea."))
R["limitaciones"] = [
    "Independiente de las etiquetas internas, no demostrado inédito para el entrenamiento de los modelos.",
    "Nemotron declara MIRACL y SQuAD entre sus fuentes de entrenamiento; no se afirma contaminación del split evaluado.",
    "Los autores de modelos pueden haber utilizado estos benchmarks durante desarrollo; comparar no demuestra generalización inédita.",
    "Qrels publicados posiblemente incompletos: documentos no anotados aportan cero por convención, no por revisión nueva.",
    "SCIDOCS usa vínculos de citación como referencia; no son etiquetas directas de pertinencia para coautoría.",
    "XQuAD es adaptación de QA a recuperación del pasaje de origen; corpus de 240 pasajes, no búsqueda abierta.",
    "MIRACL HN español usa el corpus reducido publicado; no es MIRACL completo.",
    "Los CI dependen de los bloques y no capturan todos los errores sistemáticos; no garantizan certeza absoluta.",
    "Las precisiones/pooling/prefijos son configuraciones específicas, no comparación abstracta de arquitecturas.",
    "No se evalúa generación, abstención, coexistencia con Gemma ni explicación causal del predictor de coautoría.",
]
R["estado"] = "completo"; R["finalizado_utc"] = ahora()
guardar(SALIDA, R)
display(pd.DataFrame(resumen_modelos)[["modelo", "ndcg10_macro_ponderado"]].round(5))
display(pivot.round(5))
display(pd.DataFrame(comparaciones).round(5))
print(R["comparacion"]["conclusion"])
print("No descartados:", no_descartados)
print("Comparte únicamente:", SALIDA)

## Fuentes y alcance

- BEIR: https://github.com/beir-cellar/beir — corpus, queries, qrels, splits y checksums de SciFact/SCIDOCS/NFCorpus.
- SciFact: https://github.com/allenai/scifact — afirmaciones científicas y abstracts con evidencia.
- MIRACL HN: https://huggingface.co/datasets/mteb/MIRACLRetrievalHardNegatives — corpus reducido publicado por MTEB.
- XQuAD: https://github.com/google-deepmind/xquad — 240 párrafos y 1,190 preguntas paralelas con traducción profesional.
- Modelos: https://huggingface.co/Qwen/Qwen3-Embedding-4B ; https://huggingface.co/nvidia/Nemotron-3-Embed-1B-BF16 ; https://huggingface.co/Snowflake/snowflake-arctic-embed-l-v2.0 ; https://huggingface.co/perplexity-ai/pplx-embed-v1-4b ; https://huggingface.co/microsoft/harrier-oss-v1-0.6b

**No hace falta que tú anotes:** las etiquetas proceden de las fuentes. Eso no significa que las etiquetas carezcan de trabajo humano previo ni que sean infalibles. No se usa un LLM juez ni se fabrica un conjunto de respuestas correctas con el propio candidato. No usar estos test para entrenar o ajustar instrucciones y después llamarlos validación independiente.

La validez de estructura y lógica se ha comprobado con datos de prueba controlados. Las descargas completas y la ejecución de los cinco codificadores se deben verificar en tu equipo; esta libreta no contiene resultados de GPU generados por el asistente.
**Nota de integridad MIRACL HN:** su corpus de candidatos puede omitir documentos presentes en los juicios publicados. La libreta excluye de la evaluación las consultas con positivos ausentes, descarta solamente juicios grado 0 fuera del corpus y registra cantidades exactas en `fuente.control_cobertura`. Los resultados así calculados no equivalen a los del benchmark MIRACL original completo.
